In [5]:
import pandas as pd
import csv
import math
import random
import igraph
import datetime
import numpy as np
import pandas as pd
from igraph import *
from sklearn import metrics
from collections import Counter
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option('mode.chained_assignment', None)
import warnings 
warnings.filterwarnings('ignore')

,citing,citing_assignees,cited,cited_assignees
0,CN105844614B,广东工业大学,CN105260710B,北京石油化工学院
1,CN104608125B,精工爱普生株式会社,CN105234938B,精工爱普生株式会社
2,CN106239505B,广东电网有限责任公司电力科学研究院,CN105234938B,精工爱普生株式会社
3,CN108972536B,西门子（中国）有限公司,CN105234938B,精工爱普生株式会社
4,CN109154276B,维斯塔斯风力系统集团公司,CN105257475B,北京科诺伟业科技股份有限公司;科诺伟业风能设备（北京）有限公司
...,...,...,...,...
54350,CN111727108B,欧姆龙株式会社,CN110325327B,西门子股份公司;加利福尼亚大学董事会
54351,CN109464125B,四川大学华西第二医院,CN108471946B,珀迪迈垂克斯公司
54352,CN109859159B,浙江大学,CN110073404B,南坦生物组学有限责任公司
54353,CN110355755B,深圳铭杰医疗科技有限公司,CN108942918B,沈阳建筑大学


# 专利权人自引网络问题

In [6]:
an_data = pd.read_excel("./results/最终数据/AN_data.xlsx")
an_data

,citing,citing_assignees,cited,cited_assignees
0,CN105844614B,广东工业大学,CN105260710B,北京石油化工学院
1,CN104608125B,精工爱普生株式会社,CN105234938B,精工爱普生株式会社
2,CN106239505B,广东电网有限责任公司电力科学研究院,CN105234938B,精工爱普生株式会社
3,CN108972536B,西门子（中国）有限公司,CN105234938B,精工爱普生株式会社
4,CN109154276B,维斯塔斯风力系统集团公司,CN105257475B,北京科诺伟业科技股份有限公司;科诺伟业风能设备（北京）有限公司
...,...,...,...,...
54350,CN111727108B,欧姆龙株式会社,CN110325327B,西门子股份公司;加利福尼亚大学董事会
54351,CN109464125B,四川大学华西第二医院,CN108471946B,珀迪迈垂克斯公司
54352,CN109859159B,浙江大学,CN110073404B,南坦生物组学有限责任公司
54353,CN110355755B,深圳铭杰医疗科技有限公司,CN108942918B,沈阳建筑大学


In [7]:
data = pd.DataFrame()
data['cited_a'] = an_data['cited_assignees']
data['citing_a'] = an_data['citing_assignees']

# 作者引用网络关系(不删除自引用的数据)
cited_citing = []
for index in data.index:
    cited_a,citing_a = data.iloc[index]['cited_a'], data.iloc[index]['citing_a']
    for i in cited_a.split(";"):
        for j in citing_a.split(";"):
            cited_citing.append((i,j))

In [9]:
from collections import Counter
cc_dict = dict(Counter(cited_citing))
pd_df = pd.DataFrame.from_dict(cc_dict,orient='index',columns=['count']).reset_index()

edges_a = pd_df['index'].tolist()
edges_w = pd_df['count'].tolist()

assignee_g = Graph.TupleList(edges_a,vertex_name_attr='name',edge_attrs=None,directed=True)
an_pr = pd.DataFrame([{"cited":p[0]['name'], 'AN_PR':p[1]} for p in zip(assignee_g.vs, assignee_g.pagerank(damping=0.5))])



In [11]:
an_prw = pd.DataFrame([{"cited":p[0]['name'], 'AN_PRW':p[1]} for p in zip(assignee_g.vs, assignee_g.pagerank(damping=0.5,weights=edges_w))])
an_prw

,cited,AN_PRW
0,北京石油化工学院,0.000069
1,广东工业大学,0.001041
2,精工爱普生株式会社,0.000133
3,广东电网有限责任公司电力科学研究院,0.000355
4,西门子（中国）有限公司,0.000077
...,...,...
10205,昆山美卓智能科技有限公司,0.000150
10206,整形工具股份有限公司,0.000071
10207,上海皓桦科技股份有限公司,0.000082
10208,珀迪迈垂克斯公司,0.000064


In [42]:
pd_df.to_excel("./results/tmp_data.xlsx",index=None)

In [43]:
pd_df[pd_df['index']==('北京小米移动软件有限公司','北京小米移动软件有限公司')]

,index,count
1228,"(北京小米移动软件有限公司, 北京小米移动软件有限公司)",12


In [ ]:
(锐捷网络股份有限公司, 锐捷网络股份有限公司)	8

In [45]:
edges_1 = pd.read_excel("./results/最终数据/train_data.xlsx",sheet_name=0)


In [48]:
edges_2 = pd.read_excel("./results/最终数据/train_data.xlsx",sheet_name=1)


In [46]:
citing = edges_1['citing'].tolist()
cited = edges_1['cited'].tolist()
edges = []
for p in zip(citing,cited):
    edges.append((p[0],p[1]))

g = Graph.TupleList(edges,vertex_name_attr='name',edge_attrs=None,directed=True)

In [53]:
# 找到强连通分量
strong_components = g.components(mode='STRONG')

# 找到包含环的节点
cycles = [component for component in strong_components if len(component) > 1]

# 打印包含环的节点
for i, cycle in enumerate(cycles):
    print(f"Cycle {i+1}: {cycle}")

Cycle 1: [36066, 43300]
Cycle 2: [16563, 16564]
Cycle 3: [7367, 7369]
Cycle 4: [1836, 7095]
Cycle 5: [16403, 42309]


In [60]:
g.vs[16403]['name'],g.vs[42309]['name']

('CN107423571B', 'CN108172291B')

In [49]:
pr_df = pd.DataFrame([{"citing":p[0]['name'], 'PR':p[1]} for p in zip(g.vs, g.pagerank(damping=0.5))])
new_df = edges_2.copy()
pa = new_df.groupby('citing').count()
pa = pa.reset_index()
pa_df = pa[['citing','year']]
pa_df.columns = ['citing','node_count']
pa_df

,citing,node_count
0,CN100334460C,1
1,CN100334837C,1
2,CN100334838C,1
3,CN100334841C,1
4,CN100335002C,1
...,...,...
46530,CN1996879B,1
46531,CN1997903B,1
46532,CN1997905B,1
46533,CN1997906B,1


In [50]:
pr_merge = pd.merge(new_df, pa_df, on='citing')
pr_merge_1 = pd.merge(pr_merge, pr_df, on='citing')
sum_pr = pr_merge_1.groupby("cited").sum()
sum_pr = sum_pr.reset_index()
pr_data = sum_pr[['cited','PR']]
pr_data

,cited,PR
0,12西格玛控股有限公司,0.000039
1,3M创新有限公司,0.000228
2,3WIN股份有限公司,0.000020
3,3形状股份有限公司,0.000041
4,460医药公司,0.000015
...,...,...
10205,龙迅半导体（合肥）股份有限公司,0.000023
10206,龙马智芯（珠海横琴）科技有限公司,0.000033
10207,龟田俊忠,0.000026
10208,（株）赛丽康,0.000023


In [52]:
pr_merge_1['spr_w'] = pr_merge_1['PR']/pr_merge_1['node_count']
sum_prw = pr_merge_1.groupby("cited").sum()
sum_prw = sum_prw.reset_index()
prw_data = sum_prw[['cited','spr_w']]
prw_data

,cited,spr_w
0,12西格玛控股有限公司,0.000039
1,3M创新有限公司,0.000228
2,3WIN股份有限公司,0.000020
3,3形状股份有限公司,0.000041
4,460医药公司,0.000008
...,...,...
10205,龙迅半导体（合肥）股份有限公司,0.000023
10206,龙马智芯（珠海横琴）科技有限公司,0.000033
10207,龟田俊忠,0.000026
10208,（株）赛丽康,0.000023


In [71]:
SPRW_merge = pd.merge(test_df,prw_data,on='cited')
SPRW_merge['SPRw_rank'] = SPRW_merge['spr_w'].rank(axis=0,method='min',ascending=False)

NameError: name 'test_df' is not defined

In [70]:
prw_data[prw_data['cited']=='深圳硅基智能科技有限公司']

,cited,spr_w
7739,深圳硅基智能科技有限公司,0.000101


In [67]:
pr_merge_1[pr_merge_1['citing']=='CN108172291B']

,citing,cited,Assignee_type,Assignee_score,order,year,node_count,PR,spr_w
31601,CN108172291B,深圳硅基智能科技有限公司,E,2,1,2020,1,0.000045,0.000045
